# Excel Full Tabs to PowerPoint (One Tab -> One Slide)

This notebook exports each worksheet tab as a full-fidelity PDF page (including charts/visuals), converts each page to an image, and places one tab per PowerPoint slide.

In [ ]:
# Step 1: Install required packages (run once if needed).
# This workflow requires Windows + Microsoft Excel desktop for full tab fidelity.
# !pip install python-pptx pywin32 pymupdf

In [ ]:
# Step 2: Import libraries.
from pathlib import Path
import platform
import tempfile

import fitz  # PyMuPDF
from pptx import Presentation
from pptx.util import Inches, Pt

import win32com.client as win32

In [ ]:
# Step 3: Configure file paths and runtime options.
# Works when running notebook from either /Sandbox or /Sandbox/Scripts.
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'Scripts' else cwd
scripts_dir = project_root / 'Scripts'

excel_path = scripts_dir / 'sample excel termplate.xlsx'
output_pptx_path = scripts_dir / 'excel_full_tabs_output.pptx'

# Optional allowlist of tabs to include (exact tab names).
# Leave empty to include all tabs.
selected_tabs = [
    # 'Summary',
    # 'Dashboard',
]

# Image quality for PDF-to-image conversion (2.0 = good balance).
render_zoom = 2.0

print(f'Excel source: {excel_path.resolve()}')
print(f'PowerPoint output: {output_pptx_path.resolve()}')

In [ ]:
# Step 4: Validate environment and inputs.
if platform.system() != 'Windows':
    raise EnvironmentError(
        'This notebook needs Windows + desktop Excel for full worksheet visual export. '
        'On Mac, export each worksheet to PDF manually and then import those pages into PPT.'
    )

if not excel_path.exists():
    raise FileNotFoundError(f'Excel file not found: {excel_path}')

In [ ]:
# Step 5: Export each worksheet to a single-page PDF.
# This preserves full tab visuals (tables, charts, shapes, formatting).
def export_sheets_to_pdf(excel_file, temp_dir, include_tabs=None):
    excel = win32.DispatchEx('Excel.Application')
    excel.Visible = False
    excel.DisplayAlerts = False

    workbook = None
    exports = []

    try:
        workbook = excel.Workbooks.Open(str(excel_file.resolve()))

        # Resolve sheet list in workbook order, or use allowlist order if provided.
        workbook_sheet_names = [workbook.Worksheets(i).Name for i in range(1, workbook.Worksheets.Count + 1)]

        if include_tabs:
            missing = [t for t in include_tabs if t not in workbook_sheet_names]
            if missing:
                raise ValueError(f'These tabs were not found: {missing}')
            sheet_names = include_tabs
        else:
            sheet_names = workbook_sheet_names

        for idx, sheet_name in enumerate(sheet_names, start=1):
            ws = workbook.Worksheets(sheet_name)

            # Force fit-to-one-page so each tab maps to one slide.
            ws.PageSetup.Zoom = False
            ws.PageSetup.FitToPagesWide = 1
            ws.PageSetup.FitToPagesTall = 1

            pdf_path = Path(temp_dir) / f'{idx:03d}_{sheet_name}.pdf'
            ws.ExportAsFixedFormat(0, str(pdf_path))
            exports.append((sheet_name, pdf_path))

        return exports

    finally:
        if workbook is not None:
            workbook.Close(SaveChanges=False)
        excel.Quit()

In [ ]:
# Step 6: Convert each sheet PDF to PNG image (first page only).
def pdf_page_to_png(pdf_path, png_path, zoom=2.0):
    doc = fitz.open(str(pdf_path))
    try:
        page = doc.load_page(0)
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, alpha=False)
        pix.save(str(png_path))
    finally:
        doc.close()

In [ ]:
# Step 7: Add one slide per tab image and save PowerPoint.
def add_image_slide(prs, title_text, image_path):
    slide = prs.slides.add_slide(prs.slide_layouts[6])

    # Title area
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.2), Inches(12.3), Inches(0.6))
    title_frame = title_box.text_frame
    title_frame.text = title_text
    title_frame.paragraphs[0].runs[0].font.size = Pt(24)

    # Content area below title
    left = Inches(0.4)
    top = Inches(1.0)
    max_width = Inches(12.5)
    max_height = Inches(5.9)

    pic = slide.shapes.add_picture(str(image_path), left, top, width=max_width)

    # Reinsert using height constraint if width-based insert is too tall.
    if pic.height > max_height:
        pic._element.getparent().remove(pic._element)
        slide.shapes.add_picture(str(image_path), left, top, height=max_height)


prs = Presentation()

with tempfile.TemporaryDirectory() as temp_dir:
    exported_pdfs = export_sheets_to_pdf(excel_path, temp_dir, selected_tabs if selected_tabs else None)

    for i, (sheet_name, pdf_path) in enumerate(exported_pdfs, start=1):
        png_path = Path(temp_dir) / f'{i:03d}_{sheet_name}.png'
        pdf_page_to_png(pdf_path, png_path, zoom=render_zoom)
        add_image_slide(prs, sheet_name, png_path)

output_pptx_path.parent.mkdir(parents=True, exist_ok=True)
prs.save(output_pptx_path)

print(f'Created PowerPoint: {output_pptx_path.resolve()}')